In [2]:
!pip install -q rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.1/37.1 MB 31.6 MB/s eta 0:00:0000:0100:01m


In [ ]:
import os
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from tqdm.auto import tqdm

# 1.CONFIGURATION
TRAIN_IN  = 'path of your train.csv'
TEST_IN   = 'path to your test.csv'
OUT_DIR   = '/kaggle/working/'

RADIUS = 3
N_BITS = 2048

os.makedirs(OUT_DIR, exist_ok=True)

# Initialize the modern RDKit generator
morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=RADIUS, fpSize=N_BITS)

#2.FINGERPRINT FUNCTION
def generate_ecfp_and_targets(df, smiles_col='smiles', target_col='labels'):
    fps_list = []
    valid_targets = []
    
    # Drop NaNs in critical columns before processing
    df = df.dropna(subset=[smiles_col, target_col]).reset_index(drop=True)
    
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Generating ECFP"):
        mol = Chem.MolFromSmiles(str(row[smiles_col]))
        if mol:
            fp_array = morgan_gen.GetFingerprintAsNumPy(mol)
            fps_list.append(fp_array)
            valid_targets.append(row[target_col])
        else:
            print(f"Warning: Invalid SMILES at index {idx}")
            
    X = np.array(fps_list, dtype=np.int8)
    y = np.array(valid_targets, dtype=np.float32)
    
    return X, y

#3EXECUTE PIPELINE
print("Loading datasets...")
train_df = pd.read_csv(TRAIN_IN)
test_df  = pd.read_csv(TEST_IN)

print("\nProcessing Training Set..")
X_train, y_train = generate_ecfp_and_targets(train_df)

print("\nProcessing Test Set..")
X_test, y_test = generate_ecfp_and_targets(test_df)

#SAVE OUTPUTS
print("\nSaving .npy files...")
np.save(os.path.join(OUT_DIR, 'X_train_ecfp.npy'), X_train)
np.save(os.path.join(OUT_DIR, 'X_test_ecfp.npy'), X_test)
np.save(os.path.join(OUT_DIR, 'y_train_potency.npy'), y_train)
np.save(os.path.join(OUT_DIR, 'y_test_potency.npy'), y_test)

print("Saving .csv files...")
bit_cols = [f'Bit_{i}' for i in range(N_BITS)]

# Train CSV
train_csv = pd.DataFrame(X_train, columns=bit_cols)
train_csv.insert(0, 'labels', y_train)
train_csv.to_csv(os.path.join(OUT_DIR, 'X_train_ecfp.csv'), index=False)

# Test CSV
test_csv = pd.DataFrame(X_test, columns=bit_cols)
test_csv.insert(0, 'labels', y_test)
test_csv.to_csv(os.path.join(OUT_DIR, 'X_test_ecfp.csv'), index=False)

print(f"\n Success Processed {len(X_train)} train and {len(X_test)} test samples.")